In [ ]:
import re
import torch
import pandas as pd
from tqdm import tqdm
from huggingface_hub import login
from transformers import AutoTokenizer, Gemma3ForConditionalGeneration, BitsAndBytesConfig


login()


input_path = "/content/haber_kaynagi_high_low_cumle_soru_eklenmis.csv"
output_path = "/content/gemma_3_27b_it_results.csv"

df = pd.read_csv(input_path)



print("Veri boyutu:", df.shape)
print(df.head())


model_id = "google/gemma-3-27b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

model.eval()

print("Model yüklendi:", model_id)


def make_prompt(cumle, soru):
    return f"""
Sana bazı cümleler verilecektir.
Her cümleden sonra, o cümleyle ilgili bir soru göreceksin.
Görevin, verilen soruya göre 1 ile 7 arasında bir değerlendirme yapmaktır.

1 = çok düşük
7 = çok yüksek

Cümle: "{cumle}"
Soru: {soru}

Sadece 1 ile 7 arasında tek bir sayı ver.

SADECE CEVAP ÜRET VE AÇIKLAMA ASLA YAPMA SORUYU TEKRAR ETME

Cevap:
""".strip()


def extract_score(text):
    """
    Model çıktısından 1-7 arasında skor çeker.
    Önce 'Skor: 5' formatını arar.
    Bulamazsa metindeki ilk 1-7 arasındaki tek sayıyı alır.
    """
    match = re.search(r"Skor\s*:\s*([1-7])", text)
    if match:
        return int(match.group(1))

    match = re.search(r"\b[1-7]\b", text)
    if match:
        return int(match.group())

    return None

def get_model_score(cumle, soru):
    prompt = make_prompt(cumle, soru)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.1,
            top_p=1,
            pad_token_id=tokenizer.eos_token_id
        )


    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    score = extract_score(response)

    return score, response


sample = df.iloc[0]

test_score, test_raw = get_model_score(
    sample["cumle"],
    sample["cumlenin_sorusu"]
)

print("\nÖrnek test")
print("Cümle:", sample["cumle"])
print("Soru:", sample["cumlenin_sorusu"])
print("Model cevabı:", test_raw)
print("Çıkarılan skor:", test_score)


all_results = []

for tur in range(3):
    print(f"\n=== TUR {tur + 1} ===")

    df_shuffled = df.sample(
        frac=1,
        random_state=42 + tur
    ).reset_index(drop=True)

    scores = []
    raw_outputs = []

    for _, row in tqdm(df_shuffled.iterrows(), total=len(df_shuffled)):
        score, raw = get_model_score(
            row["cumle"],
            row["cumlenin_sorusu"]
        )

        scores.append(score)
        raw_outputs.append(raw)

    df_shuffled["tur"] = tur + 1
    df_shuffled["gemma_3_27b_it_score"] = scores
    df_shuffled["gemma_3_27b_it_raw"] = raw_outputs

    all_results.append(df_shuffled)

df_results = pd.concat(all_results, ignore_index=True)

print("\nSonuç önizleme:")
display(df_results.head())

df_results.to_csv(output_path, index=False)

print("\nSonuç dosyası kaydedildi:")
print(output_path)